# QariOCR FT2 - Dual Model Training
## Extraction Model + Verification Model

**Optimized for**: Kaggle Notebooks (Tesla T4 x2 - 30GB total memory)
**Base Model**: Qwen2-VL-2B-Instruct (4-bit quantized)
**Training Time**: 4-6 hours total on Kaggle T4 x2

---

### Pre-requisites:

1. **Enable GPU**: Settings → Accelerator → **GPU T4 x2** (30GB memory)
2. **Add Dataset**: Add `qari-ocr-ft2` dataset to this notebook
3. **Run**: Execute cells sequentially

### FT2 Key Changes:
- **40% Correct / 60% Errors** - Error-focused dataset
- **No Auto-Correction** - Models trained to extract exactly what's visible
- **Data Augmentation** - 1,208 augmented perfect pages
- **Diverse Errors** - 800 synthetic errors across 4 categories
- **Dual Purpose** - Separate extraction & verification models

### 1. Installation & Setup

In [ ]:
# Install Unsloth and dependencies
import os
# Force single GPU to avoid multi-GPU overhead in Kaggle notebooks
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Using single GPU: CUDA_VISIBLE_DEVICES=0")
import subprocess
import sys

print("🔄 Installing required packages...")
packages = [
    "unsloth",
    "transformers>=4.56.1",
    "trl==0.22.2",
    "datasets",
    "accelerate",
    "bitsandbytes"
]

for package in packages:
    try:
        if package == "trl==0.22.2":
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", package])
        else:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} installed")
    except Exception as e:
        print(f"⚠️ {package}: {e}")

print("\n✅ Installation complete!")

### 2. Import Libraries

In [ ]:
import os
import json
import torch
torch.backends.cudnn.benchmark = True
from PIL import Image
from datasets import Dataset
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import gc

print("✅ Libraries imported")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

### 3. Data Loading Function

In [ ]:
# Load FT2 dataset
def load_ft2_dataset(json_file, base_path, is_extraction=True):
    print(f"Loading {json_file}...")
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    formatted_samples = []
    missing_images = 0
    
    for i, item in enumerate(data):
        try:
            # Get image path from the item
            img_path_rel = item['image_path']  # e.g., "synthetic_errors_v2_full/error_v2_character_553_2491.jpg"
            img_path = os.path.join(base_path, img_path_rel)
            
            if not os.path.exists(img_path):
                missing_images += 1
                continue
            
            # Create prompts based on task type
            if is_extraction:
                # Extraction task: Extract text exactly as shown
                user_text = "Extract the Quranic text from this image EXACTLY as it appears. DO NOT auto-correct any errors. DO NOT fill in missing characters. Report exactly what you see, character by character."
                assistant_text = item['error_text'] if item['has_error'] else item['ground_truth_text']
            else:
                # Verification task: Detect and report errors
                user_text = "Analyze this Quranic text image and identify any errors. Report the exact errors found, including missing characters, wrong diacritics, extra characters, or any variations from standard text."
                if item['has_error']:
                    assistant_text = f"ERROR DETECTED: {item['error_type']} - {item.get('error_info', {}).get('description', 'Error in text')}"
                else:
                    assistant_text = "No errors detected. Text is correct."
            
            formatted_samples.append({
                "image_path": img_path,
                "user_prompt": user_text,
                "assistant_response": assistant_text
            })
            
        except Exception as e:
            missing_images += 1
            continue
    
    print(f"✅ Loaded {len(formatted_samples)} samples (skipped {missing_images})")
    return formatted_samples

print("✅ Data loading function defined")

### 4. Load FT2 Datasets

In [ ]:
# Define BASE_PATH first
BASE_PATH = "/kaggle/input/qari-ocr-ft2"
print(f"📁 Using dataset path: {BASE_PATH}")

# Load extraction datasets (is_extraction=True)
train_extraction = load_ft2_dataset(f"{BASE_PATH}/ft2_extraction/train.json", BASE_PATH, is_extraction=True)
val_extraction = load_ft2_dataset(f"{BASE_PATH}/ft2_extraction/val.json", BASE_PATH, is_extraction=True)

print(f"\n📊 Extraction Dataset:")
print(f"   Train: {len(train_extraction)}")
print(f"   Val: {len(val_extraction)}")

In [ ]:
# Debug: Check JSON file structure
import os

BASE_PATH = "/kaggle/input/qari-ocr-ft2"
TRAIN_JSON = f"{BASE_PATH}/ft2_extraction/train.json"

print(f"Checking {TRAIN_JSON}...")
print(f"File exists: {os.path.exists(TRAIN_JSON)}")

if os.path.exists(TRAIN_JSON):
    with open(TRAIN_JSON, 'r') as f:
        data = json.load(f)
    print(f"Total items in JSON: {len(data)}")
    if len(data) > 0:
        print(f"First item keys: {data[0].keys()}")
        print(f"First item structure:\n{json.dumps(data[0], indent=2, ensure_ascii=False)[:500]}")
else:
    print("❌ JSON file not found!")
    print(f"Checking directory structure...")
    print(f"BASE_PATH exists: {os.path.exists(BASE_PATH)}")
    if os.path.exists(BASE_PATH):
        print(f"Contents: {os.listdir(BASE_PATH)[:10]}")

print("\n" + "="*70)
print("Checking image directories...")
print(f"Augmented perfect exists: {os.path.exists(f'{BASE_PATH}/augmented_perfect')}")
print(f"Synthetic errors exists: {os.path.exists(f'{BASE_PATH}/synthetic_errors_v2_full')}")
if os.path.exists(f'{BASE_PATH}/augmented_perfect'):
    aug_files = os.listdir(f'{BASE_PATH}/augmented_perfect')
    print(f"Augmented images count: {len(aug_files)}")
    print(f"First 3 files: {aug_files[:3]}")
if os.path.exists(f'{BASE_PATH}/synthetic_errors_v2_full'):
    err_files = os.listdir(f'{BASE_PATH}/synthetic_errors_v2_full')
    print(f"Error images count: {len(err_files)}")
    print(f"First 3 files: {err_files[:3]}")



In [ ]:
# Define BASE_PATH first
BASE_PATH = "/kaggle/input/qari-ocr-ft2"
print(f"📁 Using dataset path: {BASE_PATH}")

# Load verification datasets (is_extraction=False)
train_verification = load_ft2_dataset(f"{BASE_PATH}/ft2_verification/train.json", BASE_PATH, is_extraction=False)
val_verification = load_ft2_dataset(f"{BASE_PATH}/ft2_verification/val.json", BASE_PATH, is_extraction=False)

print(f"\n📊 Verification Dataset:")
print(f"   Train: {len(train_verification)}")
print(f"   Val: {len(val_verification)}")

### 5. Convert to Conversation Format

In [ ]:
def convert_to_conversation(sample):
    try:
        # Load image as PIL Image object
        image = Image.open(sample["image_path"]).convert("RGB")
        
        # Use the format for Unsloth vision data collator
        conversation = [
            { 
                "role": "user",
                "content": [
                    {"type": "text", "text": sample["user_prompt"]},
                    {"type": "image"}  # Don't pass image here
                ]
            },
            { 
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["assistant_response"]}
                ]
            },
        ]
        # Return with image and messages separately
        return {"messages": conversation, "images": [image]}
    except Exception as e:
        print(f"⚠️ Error processing sample {sample.get('image_path', 'unknown')}: {e}")
        return None

print("✅ Conversion function defined")

In [ ]:
# Convert extraction datasets - keep PIL images in memory
print("🔄 Converting extraction datasets...")
converted_train_extraction = []
for sample in train_extraction:
    conv = convert_to_conversation(sample)
    if conv:
        converted_train_extraction.append(conv)

converted_val_extraction = []
for sample in val_extraction:
    conv = convert_to_conversation(sample)
    if conv:
        converted_val_extraction.append(conv)

train_extraction_ds = Dataset.from_list(converted_train_extraction)
val_extraction_ds = Dataset.from_list(converted_val_extraction)

print(f"✅ Extraction datasets converted: {len(train_extraction_ds)} train, {len(val_extraction_ds)} val")

In [ ]:
# Convert verification datasets
print("🔄 Converting verification datasets...")
converted_train_verification = []
for sample in train_verification:
    conv = convert_to_conversation(sample)
    if conv:
        converted_train_verification.append(conv)

converted_val_verification = []
for sample in val_verification:
    conv = convert_to_conversation(sample)
    if conv:
        converted_val_verification.append(conv)

train_verification_ds = Dataset.from_list(converted_train_verification)
val_verification_ds = Dataset.from_list(converted_val_verification)

print(f"✅ Verification datasets converted: {len(train_verification_ds)} train, {len(val_verification_ds)} val")

In [10]:
# Lazy path-to-PIL collator: converts string paths to PIL on-the-fly
from PIL import Image
from unsloth.trainer import UnslothVisionDataCollator

class LazyPathVisionCollator:
    def __init__(self, model, tokenizer):
        self.inner = UnslothVisionDataCollator(model, tokenizer)

    def __call__(self, features):
        for f in features:
            if "images" in f and isinstance(f["images"], list):
                out_imgs = []
                for img in f["images"]:
                    if isinstance(img, str):
                        out_imgs.append(Image.open(img).convert("RGB"))
                    else:
                        out_imgs.append(img)
                f["images"] = out_imgs
        return self.inner(features)

### 6. Training Function

In [ ]:
def train_model(train_ds, val_ds, model_type):
    print(f"\n{'='*70}")
    print(f"🚀 Starting {model_type.upper()} Model Training")
    print(f"{'='*70}\n")
    
    # Clear GPU memory
    gc.collect()
    torch.cuda.empty_cache()
    
    # Load model
    print("📥 Loading base model...")
    model, tokenizer = FastVisionModel.from_pretrained(
        "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth"
    )
    
    # Apply LoRA
    print("🔧 Applying LoRA...")
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=True,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=24,
        lora_alpha=24,
        lora_dropout=0,
        bias="none",
        random_state=3407
    )
    
    # CRITICAL: Enable training mode BEFORE creating trainer
    FastVisionModel.for_training(model)
    
    # Train
    print("👨‍🏫 Configuring training...")
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=SFTConfig(
            per_device_train_batch_size=4,          # was 2
            per_device_eval_batch_size=2,
            gradient_accumulation_steps=2,          # was 4 (global batch stays ~8)
            num_train_epochs=5,  # Full training
            learning_rate=1.5e-4,
            warmup_steps=50,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            logging_steps=20,                        # was 50
            eval_strategy="no",
            save_strategy="no",
            output_dir=f"/kaggle/working/FT2_{model_type}",
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_length=2048,
            seed=3407,
            fp16=True,
            dataloader_num_workers=2,                # was 0
            dataloader_pin_memory=True,
            dataloader_persistent_workers=True,
            dataloader_prefetch_factor=2,            # add this
            report_to="none",
        )
    )
    
    print(f"\n🏃 Training {model_type} model...")
    start_time = datetime.now()
    trainer.train()
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds() / 60
    
    # Save models with directory creation and verification
    print(f"\n💾 Saving {model_type} model...")
    import os
    output_dir_lora = f"/kaggle/working/FT2_{model_type}"
    output_dir_merged = f"/kaggle/working/FT2_{model_type}_merged"
    os.makedirs(output_dir_lora, exist_ok=True)
    os.makedirs(output_dir_merged, exist_ok=True)

    # Save LoRA adapters
    print(f"   Saving LoRA adapters to {output_dir_lora}...")
    model.save_pretrained(output_dir_lora)
    tokenizer.save_pretrained(output_dir_lora)

    # Save merged 16bit model
    print(f"   Saving merged 16bit model to {output_dir_merged}...")
    model.save_pretrained_merged(output_dir_merged, tokenizer, save_method="merged_16bit")

    # Verify files were saved
    print(f"\n   ✅ Verifying saved files...")
    lora_files = os.listdir(output_dir_lora) if os.path.exists(output_dir_lora) else []
    merged_files = os.listdir(output_dir_merged) if os.path.exists(output_dir_merged) else []
    print(f"   LoRA directory: {len(lora_files)} files")
    print(f"   Merged directory: {len(merged_files)} files")
    if lora_files:
        print(f"   LoRA files: {lora_files[:5]}")
    if merged_files:
        print(f"   Merged files: {merged_files[:5]}")
    
    print(f"✅ {model_type.upper()} Model Complete!")
    print(f"   Duration: {duration:.1f} minutes")
    
    # Cleanup
    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()
    
    return duration

print("✅ Training function defined")

### 7. Train Extraction Model

In [ ]:
extraction_duration = train_model(train_extraction_ds, val_extraction_ds, "extraction")

### 8. Train Verification Model

In [ ]:
verification_duration = train_model(train_verification_ds, val_verification_ds, "verification")

### 9. Training Summary

In [ ]:
print(f"\n{'='*70}")
print("🎉 FT2 DUAL MODEL TRAINING COMPLETE!")
print(f"{'='*70}\n")
print(f"📊 Training Summary:")
print(f"   Extraction Model: {extraction_duration:.1f} minutes")
try:
    print(f"   Verification Model: {verification_duration:.1f} minutes")
    print(f"   Total Time: {extraction_duration + verification_duration:.1f} minutes")
except NameError:
    print("   Verification Model: skipped in this run")
    print(f"   Total Time: {extraction_duration:.1f} minutes")
print(f"\n📦 Output Models:")
print("   1. FT2_extraction (LoRA + Merged)")
print("   2. FT2_verification (LoRA + Merged)")
print(f"\n✅ Download models from Kaggle Output tab!")